# Aerosols video sonification (Windows compatible)
Play `arkaplan.wav` as a looping background and mix green, blue, purple, red background, and red foreground samples according to matching pixels along a white scanline clipped to the Earth disk. The scan makes one pass from the top of Earth to the bottom over the video duration. The original video keeps its aspect ratio and timing. Run cells from top to bottom.

FFmpeg is loaded from PATH or the project-local `.notebook-deps` package. If dependencies are missing, run `%pip install numpy opencv-python imageio-ffmpeg` in a new cell, then restart the kernel. No Pixasonics or macOS tools are required.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import wave
import cv2
import numpy as np

# Find the notebook folder whether Jupyter starts here or in the workspace root.
NOTEBOOK_DIR = next((p for p in (Path.cwd(), Path.cwd() / "SpaceAppsChallengeJukeBox")
                     if (p / "color_sampled_pixasonics.ipynb").is_file()), None)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("Start Jupyter in the project root or notebook folder.")
local_packages = NOTEBOOK_DIR / ".notebook-deps"
if local_packages.is_dir():
    sys.path.insert(0, str(local_packages))
FFMPEG = shutil.which("ffmpeg")
if FFMPEG is None:
    try:
        import imageio_ffmpeg
        FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
    except (ImportError, RuntimeError) as exc:
        raise RuntimeError("Run %pip install imageio-ffmpeg in a new cell, then restart the kernel.") from exc
subprocess.run([FFMPEG, "-version"], check=True, capture_output=True)
print("FFmpeg ready:", FFMPEG)

## Inputs and sound mappings
Add future sounds to `COLOR_SOUNDS`. Hue uses OpenCV's 0–179 scale: red ≈ 0, yellow ≈ 30, green ≈ 60, blue ≈ 110. `hue_radius` controls the accepted hue range. Missing configured files produce a clear error; unfinished sounds can stay out of the list until ready.

`PREVIEW_SECONDS = 5.0` renders a quick check; use `None` for the entire video. Each color gain is proportional to the fraction of the scanline containing that color, with saturation/brightness confidence and smoothed transitions. Adjust `gain` if the sound is too quiet.

`red_background.mp3` starts at 8 seconds into its source file whenever the timeline begins; its first 8 seconds are excluded from the loop. Both red tracks follow red pixels; `arkaplan.wav` remains the continuous background. `start_seconds` can be set separately for any sound.

Color tracks are boosted relative to the background. The final mix is RMS normalized toward `OUTPUT_TARGET_RMS_DBFS` (-18 dBFS), with `OUTPUT_PEAK_CEILING` preventing clipping. This is amplitude normalization, not LUFS normalization.

Both red tracks have half their previous gain (-6 dB), a 700 ms gain smoothing time, and a 500 ms loop crossfade. Other color settings remain unchanged.

In [ ]:
DATA_DIR = NOTEBOOK_DIR.parent / "filteredData" / "Aerosols"
VIDEO_PATH = DATA_DIR / "Aerosols_RotatingEarth_3840x2160.mp4"
BACKGROUND_PATH = DATA_DIR / "arkaplan.wav"
COLOR_SOUNDS = [
    {"name": "green", "path": DATA_DIR / "yeşil.wav", "hue": 60, "hue_radius": 25, "gain": 2.0},
    {"name": "blue", "path": DATA_DIR / "mavi.mp3", "hue": 110, "hue_radius": 25, "gain": 2.0},
    {"name": "purple", "path": DATA_DIR / "mor.wav", "hue": 150, "hue_radius": 20, "gain": 2.0},
    {"name": "red_background", "path": DATA_DIR / "red_background.mp3", "hue": 0, "hue_radius": 20, "gain": 0.7, "start_seconds": 8.0, "smoothing_ms": 700, "loop_crossfade_ms": 500},
    {"name": "red_foreground", "path": DATA_DIR / "red_foreground.mp3", "hue": 0, "hue_radius": 20, "gain": 1.0, "smoothing_ms": 700, "loop_crossfade_ms": 500},
]
PREVIEW_SECONDS = None
MAX_VIDEO_WIDTH = 1280
SAMPLE_RATE = 44100
BACKGROUND_GAIN = 0.4
OUTPUT_TARGET_RMS_DBFS = -18.0
OUTPUT_PEAK_CEILING = 0.95
SMOOTHING_MS = 180
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "aerosols_sonification.mp4"
AUDIO_PATH = OUTPUT_DIR / "aerosols_sonification.wav"
TEMP_VIDEO_PATH = OUTPUT_DIR / "aerosols_scan_silent.mp4"

missing = [p for p in [VIDEO_PATH, BACKGROUND_PATH, *(s["path"] for s in COLOR_SOUNDS)] if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(map(str, missing)))
for sound in COLOR_SOUNDS:
    if not 0 <= sound["hue"] < 180 or not 0 < sound["hue_radius"] <= 90 or sound["gain"] < 0 or sound.get("start_seconds", 0) < 0:
        raise ValueError(f"Invalid color mapping: {sound['name']}")
print("Inputs ready:", DATA_DIR)


In [ ]:
def decode_audio(path, start_seconds=0.0, loop_crossfade_ms=40):
    result = subprocess.run(
        [FFMPEG, "-v", "error", "-i", str(path), "-ss", str(start_seconds), "-f", "f32le", "-ac", "2",
         "-ar", str(SAMPLE_RATE), "pipe:1"], capture_output=True, check=True)
    audio = np.frombuffer(result.stdout, dtype="<f4").reshape(-1, 2).copy()
    if not audio.size or not np.isfinite(audio).all():
        raise ValueError(f"Empty or invalid audio: {path}")
    rms = float(np.sqrt(np.mean(audio.astype(np.float64) ** 2)))
    peak = float(np.max(np.abs(audio)))
    if rms < 1e-8:
        raise ValueError(f"Silent audio: {path}")
    audio *= min(10 ** (-24 / 20) / rms, 0.89 / peak)
    # Crossfade the end into the start to reduce clicks when looping.
    fade = min(round(SAMPLE_RATE * loop_crossfade_ms / 1000), len(audio) // 4)
    if fade >= 2:
        ramp = np.linspace(0, 1, fade, dtype=np.float32)[:, None]
        audio = np.concatenate((audio[:-fade], audio[-fade:] * (1-ramp) + audio[:fade] * ramp))
    return audio

def loop_chunk(audio, start, count):
    return audio[(np.arange(count) + start) % len(audio)]

background = decode_audio(BACKGROUND_PATH)
for sound in COLOR_SOUNDS:
    sound["audio"] = decode_audio(sound["path"], sound.get("start_seconds", 0.0), sound.get("loop_crossfade_ms", 40))
    print(f"Loaded {sound['name']}: {len(sound['audio']) / SAMPLE_RATE:.1f}s, source starts at {sound.get('start_seconds', 0.0):.1f}s")

In [ ]:
capture = cv2.VideoCapture(str(VIDEO_PATH))
writer = None
try:
    if not capture.isOpened():
        raise RuntimeError(f"Could not open {VIDEO_PATH}")
    fps = capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or frame_count <= 0:
        raise ValueError("Video has invalid FPS or frame count.")
    if PREVIEW_SECONDS is not None:
        if PREVIEW_SECONDS <= 0:
            raise ValueError("PREVIEW_SECONDS must be positive or None.")
        frame_count = min(frame_count, max(1, round(PREVIEW_SECONDS * fps)))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = min(1, MAX_VIDEO_WIDTH / width)
    size = (max(2, int(width * scale) // 2 * 2), max(2, int(height * scale) // 2 * 2))
    # Estimate the fixed globe disk from the first frame against black space.
    ok, reference = capture.read()
    if not ok:
        raise RuntimeError("Could not read the first frame for Earth detection.")
    reference = cv2.resize(reference, size, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(reference, cv2.COLOR_BGR2GRAY)
    visible = (gray > 3).astype(np.uint8) * 255
    visible = cv2.morphologyEx(visible, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8))
    contours, _ = cv2.findContours(visible, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        raise ValueError("Could not detect Earth against the black background.")
    globe = max(contours, key=cv2.contourArea)
    (earth_x, earth_y), earth_radius = cv2.minEnclosingCircle(globe)
    earth_radius = max(1.0, earth_radius - 1.0)  # Keep white pixels inside the limb.
    yy, xx = np.ogrid[:size[1], :size[0]]
    earth_mask = (xx - earth_x)**2 + (yy - earth_y)**2 <= earth_radius**2
    earth_rows = np.flatnonzero(earth_mask.any(axis=1))
    earth_top, earth_bottom = int(earth_rows[0]), int(earth_rows[-1])
    capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
    print(f"Earth scan: y={earth_top} to {earth_bottom}, radius={earth_radius:.1f}px")
    writer = cv2.VideoWriter(str(TEMP_VIDEO_PATH), cv2.VideoWriter_fourcc(*"mp4v"), fps, size)
    if not writer.isOpened():
        raise RuntimeError("Could not create output video.")
    weights = np.zeros(len(COLOR_SOUNDS), dtype=np.float32)
    blocks = []
    cursor = 0
    for index in range(frame_count):
        ok, frame = capture.read()
        if not ok:
            raise RuntimeError(f"Could not decode frame {index}/{frame_count}.")
        frame = cv2.resize(frame, size, interpolation=cv2.INTER_AREA)
        progress = index / max(1, frame_count - 1)
        y = round(earth_top + (earth_bottom - earth_top) * progress)
        row = cv2.cvtColor(frame[y:y+1], cv2.COLOR_BGR2HSV)[0].astype(np.float32)
        confidence = np.clip((row[:, 1] / 255 - 0.08) / 0.35, 0, 1)
        confidence *= np.clip((row[:, 2] / 255 - 0.04) / 0.25, 0, 1)
        end = round((index + 1) * SAMPLE_RATE / fps)
        count = end - cursor
        block = loop_chunk(background, cursor, count) * BACKGROUND_GAIN
        for j, sound in enumerate(COLOR_SOUNDS):
            smoothing_ms = sound.get("smoothing_ms", SMOOTHING_MS)
            ease = 1 - np.exp(-np.arange(1, count + 1, dtype=np.float32) / max(1, SAMPLE_RATE * smoothing_ms / 1000))
            delta = np.abs(row[:, 0] - sound["hue"])
            distance = np.minimum(delta, 180 - delta)
            target = float(np.mean(np.clip(1 - distance / sound["hue_radius"], 0, 1) * confidence))
            envelope = weights[j] + ease * (target - weights[j])
            block += loop_chunk(sound["audio"], cursor, count) * envelope[:, None] * sound["gain"]
            weights[j] = envelope[-1]
        blocks.append(block)
        cursor = end
        # Clip the complete stroke, including thickness, to the Earth disk.
        line_mask = np.zeros((size[1], size[0]), dtype=np.uint8)
        cv2.line(line_mask, (0, y), (size[0] - 1, y), 255, 2)
        stroke = (line_mask > 0) & earth_mask
        frame[stroke] = (255, 255, 255)
        if index in {0, frame_count // 2, frame_count - 1}:
            cv2.imwrite(str(OUTPUT_DIR / f"scan_check_{index:04d}.png"), frame)
        writer.write(frame)
        if index % max(1, round(fps * 10)) == 0:
            print(f"Rendered {index + 1}/{frame_count} frames")
finally:
    capture.release()
    if writer is not None:
        writer.release()

audio = np.concatenate(blocks)
peak = float(np.max(np.abs(audio)))
rms = float(np.sqrt(np.mean(np.square(audio, dtype=np.float64))))
if rms > 1e-8:
    output_gain = min(10 ** (OUTPUT_TARGET_RMS_DBFS / 20) / rms, OUTPUT_PEAK_CEILING / peak)
    audio *= output_gain
    print(f"Output volume adjustment: {20 * np.log10(output_gain):+.1f} dB")
with wave.open(str(AUDIO_PATH), "wb") as wav:
    wav.setnchannels(2)
    wav.setsampwidth(2)
    wav.setframerate(SAMPLE_RATE)
    wav.writeframes((np.clip(audio, -1, 1) * 32767).astype("<i2").tobytes())
subprocess.run([FFMPEG, "-y", "-v", "error", "-i", str(TEMP_VIDEO_PATH), "-i", str(AUDIO_PATH),
                "-map", "0:v:0", "-map", "1:a:0", "-c:v", "libx264", "-crf", "18",
                "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k", "-shortest", "-movflags", "+faststart",
                str(OUTPUT_PATH)], check=True)
print(f"Saved {frame_count / fps:.2f}s video: {OUTPUT_PATH}")
print("Saved audio:", AUDIO_PATH)